# M06-P03 — Tool Calling & Enterprise Data

Hasta ahora el asistente recuperaba documentación. En esta práctica debe consultar datos empresariales estructurados mediante **tools reales**.

```text
pregunta → LLM → toolUse → Python → SQLite → toolResult → LLM → respuesta
```

La idea clave es que el modelo **no ejecuta SQL**: decide qué herramienta necesita y nuestra aplicación controla qué código puede ejecutarse.

In [ ]:
import json
import pandas as pd

from enterprise_genai.enterprise_tools import (
    TOOL_CONFIG,
    database_overview,
    get_contract_sla,
    get_open_incidents,
    get_service_status,
    investigate_with_tools,
)

overview = database_overview()
print('Base de datos:', overview['database'])
print('Tablas y filas:')
display(pd.DataFrame([{'tabla': k, 'filas': v} for k, v in overview['tables'].items()]))

## 1. Primero: las tools son funciones normales

Antes de dejar que el LLM las elija, llamamos directamente a las tres funciones. Todas abren SQLite en modo de solo lectura y usan consultas parametrizadas.

In [ ]:
print('--- Estado del servicio ---')
display(get_service_status('Managed VPN'))

print('--- Incidentes abiertos ---')
display(get_open_incidents('Asteria Retail', 'Managed VPN'))

print('--- SLA contratado ---')
display(get_contract_sla('Asteria Retail', 'Managed VPN'))

## 2. Qué ve el modelo: schemas, no implementación

Bedrock recibe nombres, descripciones y JSON Schemas. No recibe acceso libre a Python ni a SQLite.

In [ ]:
for item in TOOL_CONFIG['tools']:
    spec = item['toolSpec']
    print('\nTOOL:', spec['name'])
    print('Descripción:', spec['description'])
    print('Schema:', json.dumps(spec['inputSchema']['json'], ensure_ascii=False, indent=2))

## 3. El LLM decide qué tools necesita

La pregunta requiere combinar tres fuentes estructuradas: incidente, estado del servicio y SLA.

In [ ]:
QUESTION = (
    'Asteria Retail informa de fallos de autenticación en Managed VPN. '
    'Investiga si hay un incidente abierto, cuál es el estado actual del servicio '
    'y qué tiempo de respuesta P1 tiene contratado. Resume la situación sin inventar datos.'
)

investigation = investigate_with_tools(QUESTION)
print(investigation['answer'])
print('\nTool calls:', len(investigation['tool_trace']))
print('Uso total:', investigation['usage'])

## 4. Abrir la caja: tool calls

Inspeccionamos qué pidió realmente el modelo, con qué argumentos y qué devolvió nuestra aplicación.

In [ ]:
trace_rows = []
for item in investigation['tool_trace']:
    trace_rows.append({
        'turn': item['turn'],
        'tool': item['name'],
        'status': item['status'],
        'input': json.dumps(item['input'], ensure_ascii=False),
        'result': json.dumps(item['result'], ensure_ascii=False),
    })
display(pd.DataFrame(trace_rows))

for item in investigation['tool_trace']:
    print(f"\n--- {item['name']} ---")
    print('INPUT:', json.dumps(item['input'], ensure_ascii=False, indent=2))
    print('RESULT:', json.dumps(item['result'], ensure_ascii=False, indent=2))

## 5. Dos turnos distintos del modelo

Normalmente el primer turno termina en `tool_use`: el modelo todavía no puede responder y solicita datos. Tras recibir `toolResult`, el siguiente turno genera la respuesta final.

In [ ]:
turn_rows = []
for turn in investigation['model_trace']:
    turn_rows.append({
        'turn': turn['turn'],
        'stop_reason': turn['stop_reason'],
        'input_tokens': turn['usage'].get('inputTokens'),
        'output_tokens': turn['usage'].get('outputTokens'),
        'total_tokens': turn['usage'].get('totalTokens'),
        'content_blocks': len(turn['content']),
    })
display(pd.DataFrame(turn_rows))

## 6. Control de ejecución

El modelo solo puede solicitar las tools de la allow-list. No exponemos una tool de `execute_sql`, y SQLite se abre en modo read-only. Esto separa dos responsabilidades:

```text
LLM        → decide qué información necesita
aplicación → decide qué herramientas existen y qué pueden hacer
```

## 7. Qué hemos construido

```text
Pregunta empresarial
        ↓
Claude + tool schemas
        ↓
toolUse
        ↓
funciones Python allow-listed
        ↓
SQLite read-only
        ↓
toolResult
        ↓
Claude
        ↓
respuesta fundamentada en datos empresariales
```


## 8. Probar la integración en la aplicación Streamlit

El notebook nos ha permitido abrir la caja y observar cada pieza por separado. El último paso consiste en comprobar que la misma capacidad ya está integrada en **Enterprise GenAI Assistant**.

Desde un terminal del Codespace ejecuta:

```bash
streamlit run app/Home.py
```

Abre la página **🔧 Investigate** y deja la pregunta de ejemplo que aparece por defecto. Pulsa **Investigar con tools**.

Comprueba que la interfaz muestra:

1. **3 tool calls** y **2 turnos del modelo**.
2. Una respuesta que incluya `INC-1043`, el incidente `P1` abierto, `Managed VPN` en estado degradado y un tiempo de respuesta P1 de `30 minutos`.
3. Las tres tools con estado `success`: `get_open_incidents`, `get_service_status` y `get_contract_sla`.
4. En **🔎 Abrir la caja: turnos, tokens y control**, un primer turno con `stop_reason = tool_use` y un segundo con `stop_reason = end_turn`.
5. En **Controles de ejecución**:

```text
sqlite_mode = read-only
arbitrary_sql_tool_exposed = false
allow_listed_tools = [
  get_service_status,
  get_open_incidents,
  get_contract_sla
]
```

Esta comprobación cierra el recorrido de la práctica:

```text
funciones aisladas
→ tool schemas
→ decisión del LLM
→ ejecución controlada
→ trazas
→ integración visible en la aplicación
```

En M06-P04 convertiremos esta secuencia en un workflow explícito con LangGraph, estado, routing y aprobación humana.